# Demand Forecasting Model (Prophet + LSTM Ensemble)
This notebook trains a demand forecasting model for each product to predict sales for the next 30 days.
It uses Prophet as a baseline and an LSTM (PyTorch Lightning) if the Prophet MAPE is > 10%.
Models are logged to MLflow.

In [1]:
# Install required libraries (uncomment if running in a fresh environment)
!pip install pandas numpy prophet torch pytorch-lightning mlflow optuna scikit-learn

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
import mlflow
import optuna
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
pl.seed_everything(42)

Seed set to 42


42

In [3]:
# 1. Data Aggregation & Preprocessing
# Path to the data file
DATA_PATH = "data/new_cleaned_retail_data_with_churn.csv"

def load_and_aggregate_data(filepath):
    print("Loading data...")
    # Load dataset
    df = pd.read_csv(filepath)
    
    # Ensure invoicedate is datetime
    df['invoicedate'] = pd.to_datetime(df['invoicedate'])
    
    # Extract date from datetime
    df['date'] = df['invoicedate'].dt.date
    
    print("Aggregating daily quantity per product...")
    # Aggregate quantity per stockcode per day
    daily_sales = df.groupby(['date', 'stockcode'])['quantity'].sum().reset_index()
    daily_sales['date'] = pd.to_datetime(daily_sales['date'])
    
    return daily_sales

daily_sales = load_and_aggregate_data(DATA_PATH)
daily_sales.head()

Loading data...
Aggregating daily quantity per product...


,date,stockcode,quantity
0,2009-12-01,10002,12
1,2009-12-01,10120,60
2,2009-12-01,10125,4
3,2009-12-01,10133,6
4,2009-12-01,10135,17


In [4]:
def prepare_product_data(df, product_id):
    """Fills missing dates with 0 and returns a continuous time series."""
    product_df = df[df['stockcode'] == product_id].copy()
    if product_df.empty:
        return product_df
    
    # Create a complete date range from min to max date
    min_date, max_date = product_df['date'].min(), product_df['date'].max()
    all_dates = pd.date_range(start=min_date, end=max_date, freq='D')
    
    # Set date as index and reindex
    product_df = product_df.set_index('date')
    product_df = product_df.reindex(all_dates, fill_value=0).reset_index()
    product_df = product_df.rename(columns={'index': 'date'})
    product_df['stockcode'] = product_id
    
    return product_df

# Example: Get top 5 selling products to test
top_products = daily_sales.groupby('stockcode')['quantity'].sum().sort_values(ascending=False).head(5).index.tolist()
print("Top products to forecast:", top_products)

Top products to forecast: ['21212', '85123A', '84077', '85099B', '17003']


In [5]:
# 2. Prophet Baseline Model
def train_prophet(product_df, test_days=30):
    # Prophet requires columns 'ds' (date) and 'y' (target)
    prophet_df = product_df[['date', 'quantity']].rename(columns={'date': 'ds', 'quantity': 'y'})
    
    # Split into train and validation (last 30 days)
    if len(prophet_df) <= test_days:
        raise ValueError("Not enough data to split into train and test")
        
    train = prophet_df.iloc[:-test_days]
    test = prophet_df.iloc[-test_days:]
    
    model = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
    model.fit(train)
    
    future = model.make_future_dataframe(periods=test_days)
    forecast = model.predict(future)
    
    # Calculate MAPE
    y_true = test['y'].values
    y_pred = forecast['yhat'].iloc[-test_days:].values
    
    # Handle zeros in denominator for MAPE
    non_zero_idx = y_true != 0
    if sum(non_zero_idx) == 0:
        mape = 0.0
    else:
        mape = mean_absolute_percentage_error(y_true[non_zero_idx], y_pred[non_zero_idx])
        
    return model, forecast, mape, y_true, y_pred

In [6]:
# 3. LSTM Model (PyTorch Lightning)
class TimeSeriesDataset(Dataset):
    def __init__(self, data, seq_length=30):
        self.data = data
        self.seq_length = seq_length
        
    def __len__(self):
        return len(self.data) - self.seq_length
        
    def __getitem__(self, idx):
        x = self.data[idx:idx+self.seq_length]
        y = self.data[idx+self.seq_length]
        return torch.FloatTensor(x), torch.FloatTensor([y])

class LSTMForecaster(pl.LightningModule):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=2, learning_rate=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.linear = nn.Linear(hidden_dim, 1)
        self.criterion = nn.MSELoss()
        
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = self.linear(lstm_out[:, -1, :])
        return out
        
    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log('train_loss', loss, prog_bar=True)
        return loss
        
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)

In [7]:
def train_lstm(product_df, test_days=30, seq_length=30):
    # Scale data
    scaler = MinMaxScaler()
    data_scaled = scaler.fit_transform(product_df[['quantity']].values)
    
    if len(data_scaled) <= test_days + seq_length:
        raise ValueError("Not enough data for LSTM")
        
    train_data = data_scaled[:-test_days]
    test_data = data_scaled[-(test_days + seq_length):]
    
    train_dataset = TimeSeriesDataset(train_data, seq_length)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
    
    model = LSTMForecaster()
    
    # Train using PyTorch Lightning (uses GPU if available in Colab)
    trainer = pl.Trainer(max_epochs=50, accelerator='auto', devices='auto', enable_progress_bar=False, enable_model_summary=False)
    trainer.fit(model, train_loader)
    
    # Predict
    model.eval()
    test_dataset = TimeSeriesDataset(test_data, seq_length)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    
    predictions = []
    with torch.no_grad():
        for x, _ in test_loader:
            pred = model(x)
            predictions.append(pred.item())
            
    # Inverse transform
    predictions = scaler.inverse_transform(np.array(predictions).reshape(-1, 1)).flatten()
    y_true = product_df['quantity'].iloc[-test_days:].values
    
    # Calculate MAPE
    non_zero_idx = y_true != 0
    if sum(non_zero_idx) == 0:
        mape = 0.0
    else:
        mape = mean_absolute_percentage_error(y_true[non_zero_idx], predictions[non_zero_idx])
        
    return model, scaler, mape, y_true, predictions

In [8]:
# 4. Ensemble and MLflow Logging
mlflow.set_experiment("Demand_Forecasting")

def train_and_log_product(product_id):
    print(f"\n--- Training for product {product_id} ---")
    product_df = prepare_product_data(daily_sales, product_id)
    
    if len(product_df) < 90:
        print(f"Skipping {product_id}: not enough data points ({len(product_df)} days).")
        return None
        
    with mlflow.start_run(run_name=f"Product_{product_id}"):
        mlflow.log_param("product_id", product_id)
        
        # 1. Train Prophet
        try:
            prophet_model, forecast, p_mape, y_true, p_pred = train_prophet(product_df)
            mlflow.log_metric("prophet_mape", p_mape)
            print(f"Prophet MAPE: {p_mape:.2%}")
        except Exception as e:
            print(f"Prophet failed: {e}")
            return None
            
        final_model_type = "Prophet"
        final_mape = p_mape
        final_pred = p_pred
        
        # 2. Check MAPE threshold, train LSTM if needed
        lstm_model_info = None
        if p_mape > 0.10:
            print("Prophet MAPE > 10%. Training LSTM...")
            try:
                lstm_model, scaler, l_mape, _, l_pred = train_lstm(product_df)
                mlflow.log_metric("lstm_mape", l_mape)
                print(f"LSTM MAPE: {l_mape:.2%}")
                
                # Ensemble (Simple Average)
                ens_pred = (p_pred + l_pred) / 2
                non_zero_idx = y_true != 0
                ens_mape = mean_absolute_percentage_error(y_true[non_zero_idx], ens_pred[non_zero_idx]) if sum(non_zero_idx) > 0 else 0
                mlflow.log_metric("ensemble_mape", ens_mape)
                print(f"Ensemble MAPE: {ens_mape:.2%}")
                
                # Select best model for this product
                best_mape = min(p_mape, l_mape, ens_mape)
                if best_mape == ens_mape:
                    final_model_type = "Ensemble"
                    final_mape = ens_mape
                    final_pred = ens_pred
                elif best_mape == l_mape:
                    final_model_type = "LSTM"
                    final_mape = l_mape
                    final_pred = l_pred
                    
                lstm_model_info = {'model': lstm_model, 'scaler': scaler}
                
            except Exception as e:
                print(f"LSTM failed: {e}")
        
        mlflow.log_param("selected_model", final_model_type)
        mlflow.log_metric("final_mape", final_mape)
        print(f"Selected Model: {final_model_type} with MAPE: {final_mape:.2%}")
        
        # Save models to MLflow
        # Note: Datasets might be large, in production you register to model registry
        # mlflow.prophet.log_model(prophet_model, artifact_path="prophet_model")
        # if lstm_model_info:
        #    mlflow.pytorch.log_model(lstm_model_info['model'], artifact_path="lstm_model")
        
        return {
            'product_id': product_id,
            'model_type': final_model_type,
            'mape': final_mape,
            'prophet_model': prophet_model,
            'lstm_info': lstm_model_info
        }

# Train for a few top products as a test
results = {}
for p in top_products[:2]: # Testing 2 products
    res = train_and_log_product(p)
    if res:
        results[p] = res


--- Training for product 21212 ---


00:23:53 - cmdstanpy - INFO - Chain [1] start processing
00:23:53 - cmdstanpy - INFO - Chain [1] done processing


Prophet MAPE: 76.54%
Prophet MAPE > 10%. Training LSTM...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
`Trainer.fit` stopped: `max_epochs=50` reached.


LSTM MAPE: 142.51%
Ensemble MAPE: 107.53%
Selected Model: Prophet with MAPE: 76.54%

--- Training for product 85123A ---


00:24:50 - cmdstanpy - INFO - Chain [1] start processing
00:24:50 - cmdstanpy - INFO - Chain [1] done processing
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Prophet MAPE: 79.39%
Prophet MAPE > 10%. Training LSTM...


`Trainer.fit` stopped: `max_epochs=50` reached.


LSTM MAPE: 61.56%
Ensemble MAPE: 68.19%
Selected Model: LSTM with MAPE: 61.56%


In [9]:
# Function ready for Nandani (Backend) to review
print("Training complete. The models are saved to MLflow.")
print("For inference, the backend will use a script similar to inference.py provided alongside this notebook.")

Training complete. The models are saved to MLflow.
For inference, the backend will use a script similar to inference.py provided alongside this notebook.
